In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import davies_bouldin_score
import seaborn as sns
import cartopy.crs as cors
from yellowbrick.cluster import SilhouetteVisualizer
from sklearn.decomposition import PCA
import glob
import os
import re 

In [4]:
def read_multi_excel(path, index_name):
    # baca semua file format xlsx
    files = glob.glob(os.path.join(path, '*.xlsx'))
    li = []
    
    for filename in files:
        # print(filename)
        
        df = pd.read_excel(filename, index_col=index_name)
        li.append(df)
    
    # concat dg index kabupaten, pastikan sama persis untuk semua file yg dibaca
    df = pd.concat(li, axis=1)
    
    return df

# fungsi untuk menggantikan unnamed 1 menjadi kolom sesuai data BPS
def process_bps_cols(new_df):
    new_header = []

    for i in range(0, len(new_df.columns)):
        # keterangan variabel
        first_row = str(new_df.iloc[0, i])
        first_row = re.sub(r"\[Metode Baru\] ", r"", first_row)
        
        # tahun yang terbaca sebagai baris kedua
        second_row = str(new_df.iloc[1, i])
        
        new_col = f"{first_row} {second_row}"
        new_header.append(new_col)

    new_df.columns = new_header
    new_df = new_df.iloc[2:] # buang dua baris pertama lama

    return new_df

In [5]:
import numpy as np

features_path = [
    'datasets/HLS', # Harapan Lama Sekolah (Tahun)
    'datasets/RLS', # Rata-rata Lama Sekolah (Tahun)
    'datasets/UHH', # Umur Harapan Hidup Saat Lahir (UHH) (Tahun)
    'datasets/Pengeluaran_Per_Kapita_PPP', # Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun)
]

bukan_kabupaten = [
    'ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'JAMBI',
    'SUMATERA SELATAN', 'BENGKULU', 'LAMPUNG', 'KEP. BANGKA BELITUNG', 'KEPULAUAN RIAU',
    'DKI JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'D I YOGYAKARTA', 'JAWA TIMUR',
    'BANTEN', 'BALI', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR', 'KALIMANTAN BARAT',
    'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'SULAWESI UTARA',
    'SULAWESI TENGAH', 'SULAWESI SELATAN', 'SULAWESI TENGGARA', 'GORONTALO', 'SULAWESI BARAT',
    'MALUKU', 'MALUKU UTARA', 'PAPUA BARAT', 'PAPUA BARAT DAYA', 'PAPUA',
    'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN'
]

# relative path dataset gabungan
dataset_gabungan = "datasets/535240039.xlsx";

def combine_features_to_df(features_path):
    index_name = 'Provinsi/Kabupaten/Kota/Indonesia'
    dfs = []

    for features in features_path:
        val_df = read_multi_excel(features, index_name)
        val_df = process_bps_cols(val_df)
        val_df = val_df.drop(index=bukan_kabupaten)

        # data sebelum pemekaran 
        df_prov_lama = val_df[~val_df.index.duplicated(keep='first')]
        # data setelah pemekaran jadi 38 provinsi
        df_prov_baru = val_df[val_df.index.duplicated(keep='first')]
        # print(df_prov_baru)
    
        # replace nilai strip dg nilai yg ada di data pasca pemekaran (naive)
        for idx in df_prov_baru.index:
            if idx in df_prov_lama.index:
                mask = df_prov_lama.loc[idx].astype(str).str.strip() == "-"
                df_prov_lama.loc[idx, mask] = df_prov_baru.loc[idx, mask].values

        val_df = df_prov_lama.copy()

        # handle missing value dg backward interpolate (tahun awal ada nan)
        val_df = val_df.replace("-", np.nan)
        val_df = val_df.astype(float).interpolate(method='linear', limit_direction='backward', axis=1)

        if os.path.isfile(dataset_gabungan):
            with pd.ExcelWriter(dataset_gabungan, mode='a', if_sheet_exists='replace') as writer:  
                val_df.to_excel(writer, sheet_name=features.split("/")[1], index=True)
        else:
            val_df.to_excel(dataset_gabungan, sheet_name=features.split("/")[1], index=True)

        dfs.append(val_df)

    return dfs

dfs = combine_features_to_df(features_path)
combined = pd.concat(dfs, axis=1, join='inner')

# buat sheet gabungan
if os.path.isfile(dataset_gabungan):
    with pd.ExcelWriter(dataset_gabungan, mode='a', if_sheet_exists='replace') as writer:  
        combined.to_excel(writer, sheet_name='Gabungan', index=True)
else:
    combined.to_excel(dataset_gabungan, sheet_name='Gabungan', index=True)


In [6]:
# df_prov_baru = ipm[0][ipm[0].index.duplicated(keep='first')]
# print(ipm[0].index[ipm[0].index.duplicated()].unique())
# ipm[0] = ipm[0][~ipm[0].index.duplicated(keep='first')]

# ipm[0]

In [7]:
# df_prov_baru = df_prov_baru.loc[:, ~df_prov_baru.astype(str).apply(lambda x: x.str.strip() == "-").any()]
# df_prov_baru

In [54]:
df = pd.read_excel('datasets/535240039.xlsx', sheet_name='Gabungan', index_col='Provinsi/Kabupaten/Kota/Indonesia')
print(df.shape)
df.tail(2)

(515, 60)


,Harapan Lama Sekolah (Tahun) 2010,Harapan Lama Sekolah (Tahun) 2011,Harapan Lama Sekolah (Tahun) 2012,Harapan Lama Sekolah (Tahun) 2013,Harapan Lama Sekolah (Tahun) 2014,Harapan Lama Sekolah (Tahun) 2015,Harapan Lama Sekolah (Tahun) 2016,Harapan Lama Sekolah (Tahun) 2017,Harapan Lama Sekolah (Tahun) 2018,Harapan Lama Sekolah (Tahun) 2019,...,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2015,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2016,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2017,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2018,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2019,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2020,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2021,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2022,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2023,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2024
Provinsi/Kabupaten/Kota/Indonesia,,,,,,,,,,,,,,,,,,,,,
Kota Jayapura,13.64,13.77,13.91,13.95,14.06,14.16,14.61,14.98,14.99,15.00,...,14249,14319,14781,14922,15176,14763,14937,15189,15272,15408
INDONESIA,11.29,11.44,11.68,12.10,12.39,12.55,12.72,12.85,12.91,12.95,...,10150,10420,10664,11059,11299,11013,11156,11479,11899,12341


In [55]:
indonesia = df.iloc[df.shape[0] - 1:] # ambil buat keperluan visualisasi nanti
df_ipm = df.iloc[:-1]
print(df_ipm.shape)
df_ipm.tail()

(514, 60)


,Harapan Lama Sekolah (Tahun) 2010,Harapan Lama Sekolah (Tahun) 2011,Harapan Lama Sekolah (Tahun) 2012,Harapan Lama Sekolah (Tahun) 2013,Harapan Lama Sekolah (Tahun) 2014,Harapan Lama Sekolah (Tahun) 2015,Harapan Lama Sekolah (Tahun) 2016,Harapan Lama Sekolah (Tahun) 2017,Harapan Lama Sekolah (Tahun) 2018,Harapan Lama Sekolah (Tahun) 2019,...,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2015,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2016,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2017,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2018,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2019,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2020,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2021,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2022,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2023,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2024
Provinsi/Kabupaten/Kota/Indonesia,,,,,,,,,,,,,,,,,,,,,
Puncak,2.84,3.21,3.73,4.04,4.13,4.47,4.48,4.66,4.93,5.19,...,5118,5181,5413,5506,5702,5378,5412,5583,5975,6363
Dogiyai,7.85,8.14,8.77,8.99,9.41,9.58,9.87,10.12,10.13,10.57,...,5120,5190,5375,5522,5709,5373,5415,5705,6182,6755
Intan Jaya,4.12,4.51,5.40,5.74,6.00,6.28,6.52,6.76,7.11,7.36,...,5015,5038,5293,5440,5593,5283,5328,5624,6140,6463
Deiyai,9.06,9.23,9.40,9.57,9.75,9.76,9.77,9.78,9.79,9.80,...,4320,4383,4597,4761,4958,4632,4673,4808,5104,5509
Kota Jayapura,13.64,13.77,13.91,13.95,14.06,14.16,14.61,14.98,14.99,15.00,...,14249,14319,14781,14922,15176,14763,14937,15189,15272,15408


In [56]:
indonesia.head()

,Harapan Lama Sekolah (Tahun) 2010,Harapan Lama Sekolah (Tahun) 2011,Harapan Lama Sekolah (Tahun) 2012,Harapan Lama Sekolah (Tahun) 2013,Harapan Lama Sekolah (Tahun) 2014,Harapan Lama Sekolah (Tahun) 2015,Harapan Lama Sekolah (Tahun) 2016,Harapan Lama Sekolah (Tahun) 2017,Harapan Lama Sekolah (Tahun) 2018,Harapan Lama Sekolah (Tahun) 2019,...,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2015,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2016,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2017,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2018,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2019,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2020,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2021,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2022,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2023,Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun) 2024
Provinsi/Kabupaten/Kota/Indonesia,,,,,,,,,,,,,,,,,,,,,
INDONESIA,11.29,11.44,11.68,12.1,12.39,12.55,12.72,12.85,12.91,12.95,...,10150,10420,10664,11059,11299,11013,11156,11479,11899,12341


In [57]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA

# minmax_normalization = MinMaxScaler()
robust = RobustScaler()

df_index_copy = df_ipm.index
df_cols_copy = df_ipm.columns

# df_ipm = pd.DataFrame(minmax_normalization.fit_transform(df_ipm))
df_ipm = pd.DataFrame(robust.fit_transform(df_ipm))
# df_ipm.columns = df_cols_copy

pca = PCA(2)
df_ipm = pd.DataFrame(pca.fit_transform(df_ipm), columns=['PC1', 'PC2'])
df_ipm.index = df_index_copy

In [52]:
df_ipm.head()

,PC1,PC2
Provinsi/Kabupaten/Kota/Indonesia,,
Simeulue,-0.000726,-5.933680
Aceh Singkil,1.297387,-4.437340
Aceh Selatan,0.163996,-6.329819
Aceh Tenggara,2.284530,-4.557585
Aceh Timur,-1.416582,-1.393234


In [ ]:
import time
kmeans_results = []
kmeans_models = []

print("K-Means Clustering")
print("-"*64)
for i in range(2, 11, 1):
    kmeans = KMeans(n_clusters=i, n_init=2500, random_state=42)
    kmeans_models.append(kmeans)

    start = time.time()
    label = kmeans.fit_predict(df_ipm)
    end = time.time()

    train_time = end - start
    kmeans_results.append(label)

    silhouette_avg = silhouette_score(df_ipm, label)
    dbi = davies_bouldin_score(df_ipm, label)

    print(f'Hasil evaluasi K-Means dengan jumlah cluster', i)
    print(f'Rata rata nilai silhoutte: {silhouette_avg:.4f}', end='\t')
    print(f'Nilai Davies-Bouldin Index: {dbi:.4f}', end='\t')
    print(f'Waktu komputasi: {train_time:.4f} detik\n')

K-Means Clustering
----------------------------------------------------------------
Hasil evaluasi K-Means dengan jumlah cluster 2
Rata rata nilai silhoutte: 0.4476	Nilai Davies-Bouldin Index: 0.8234	Waktu komputasi: 6.4851 detik

Hasil evaluasi K-Means dengan jumlah cluster 3
Rata rata nilai silhoutte: 0.5040	Nilai Davies-Bouldin Index: 0.6543	Waktu komputasi: 7.2236 detik

Hasil evaluasi K-Means dengan jumlah cluster 4
Rata rata nilai silhoutte: 0.3629	Nilai Davies-Bouldin Index: 0.8242	Waktu komputasi: 6.9121 detik

Hasil evaluasi K-Means dengan jumlah cluster 5
Rata rata nilai silhoutte: 0.3528	Nilai Davies-Bouldin Index: 0.7982	Waktu komputasi: 7.0709 detik

Hasil evaluasi K-Means dengan jumlah cluster 6
Rata rata nilai silhoutte: 0.3528	Nilai Davies-Bouldin Index: 0.7863	Waktu komputasi: 7.8284 detik

Hasil evaluasi K-Means dengan jumlah cluster 7
Rata rata nilai silhoutte: 0.3427	Nilai Davies-Bouldin Index: 0.8704	Waktu komputasi: 8.0154 detik

Hasil evaluasi K-Means dengan juml

In [74]:
from sklearn.cluster import SpectralClustering
import time

spectral_clust_results = []
spectral_clust_models = []

print("Spectral Clustering")
print("-"*64)
for i in range(2, 11, 1):
    spectral_clust = SpectralClustering(n_clusters=i, n_init=2500, random_state=42)
    spectral_clust_models.append(spectral_clust)

    start = time.time()
    label = spectral_clust.fit_predict(df_ipm)
    end = time.time()

    train_time = end - start
    spectral_clust_results.append(label)

    silhouette_avg = silhouette_score(df_ipm, label)
    dbi = davies_bouldin_score(df_ipm, label)

    print(f'Hasil evaluasi Spectral Clustering dengan jumlah cluster', i)
    print(f'Rata rata nilai silhoutte: {silhouette_avg:.4f}', end='\t')
    print(f'Nilai Davies-Bouldin Index: {dbi:.4f}', end='\t')
    print(f'Waktu komputasi: {train_time:.4f} detik\n')

Spectral Clustering
----------------------------------------------------------------
Hasil evaluasi Spectral Clustering dengan jumlah cluster 2
Rata rata nilai silhoutte: 0.7769	Nilai Davies-Bouldin Index: 0.1440	Waktu komputasi: 5.6188 detik

Hasil evaluasi Spectral Clustering dengan jumlah cluster 3
Rata rata nilai silhoutte: 0.7046	Nilai Davies-Bouldin Index: 0.2658	Waktu komputasi: 5.4523 detik

Hasil evaluasi Spectral Clustering dengan jumlah cluster 4
Rata rata nilai silhoutte: 0.5202	Nilai Davies-Bouldin Index: 0.4014	Waktu komputasi: 5.3588 detik

Hasil evaluasi Spectral Clustering dengan jumlah cluster 5
Rata rata nilai silhoutte: 0.5384	Nilai Davies-Bouldin Index: 0.4630	Waktu komputasi: 5.7952 detik

Hasil evaluasi Spectral Clustering dengan jumlah cluster 6
Rata rata nilai silhoutte: 0.4017	Nilai Davies-Bouldin Index: 0.5798	Waktu komputasi: 6.0904 detik

Hasil evaluasi Spectral Clustering dengan jumlah cluster 7
Rata rata nilai silhoutte: 0.3020	Nilai Davies-Bouldin Index:

In [73]:
from sklearn.cluster import BisectingKMeans

bisect_kmeans_results = []
bisect_kmeans_models = []

print("Bisecting K-Means Clustering")
print("-"*64)
for i in range(2, 11, 1):
    bisect_kmeans = BisectingKMeans(n_clusters=i, n_init=2500, random_state=42)
    bisect_kmeans_models.append(bisect_kmeans)

    start = time.time()
    label = bisect_kmeans.fit_predict(df_ipm)
    end = time.time()

    train_time = end - start
    bisect_kmeans_results.append(label)

    silhouette_avg = silhouette_score(df_ipm, label)
    dbi = davies_bouldin_score(df_ipm, label)

    print(f'Hasil evaluasi Bisecting K-Means dengan jumlah cluster', i)
    print(f'Rata rata nilai silhoutte: {silhouette_avg:.4f}', end='\t')
    print(f'Nilai Davies-Bouldin Index: {dbi:.4f}', end='\t')
    print(f'Waktu komputasi: {train_time:.4f} detik\n')

Bisecting K-Means Clustering
----------------------------------------------------------------
Hasil evaluasi Bisecting K-Means dengan jumlah cluster 2
Rata rata nilai silhoutte: 0.4476	Nilai Davies-Bouldin Index: 0.8234	Waktu komputasi: 6.4985 detik

Hasil evaluasi Bisecting K-Means dengan jumlah cluster 3
Rata rata nilai silhoutte: 0.4813	Nilai Davies-Bouldin Index: 0.6898	Waktu komputasi: 11.1618 detik

Hasil evaluasi Bisecting K-Means dengan jumlah cluster 4
Rata rata nilai silhoutte: 0.3488	Nilai Davies-Bouldin Index: 0.8358	Waktu komputasi: 15.5081 detik

Hasil evaluasi Bisecting K-Means dengan jumlah cluster 5
Rata rata nilai silhoutte: 0.3255	Nilai Davies-Bouldin Index: 0.8907	Waktu komputasi: 18.9593 detik

Hasil evaluasi Bisecting K-Means dengan jumlah cluster 6
Rata rata nilai silhoutte: 0.3045	Nilai Davies-Bouldin Index: 0.8856	Waktu komputasi: 22.4267 detik

Hasil evaluasi Bisecting K-Means dengan jumlah cluster 7
Rata rata nilai silhoutte: 0.3018	Nilai Davies-Bouldin Index